In [ ]:
from google.colab import drive
drive.mount('/content/drive')

<font color='tomato'><font color="#CC3D3D"><p>
# A DNN Model for Multiclass Classification

##### Import modules

In [ ]:
!pip install -q -U keras-tuner

In [ ]:
import pandas as pd
import numpy as np
import os
import random
import pickle
from tqdm import tqdm
from IPython.display import Image, clear_output
import seaborn as sns
import matplotlib.pylab as plt
from matplotlib import font_manager, rc
%matplotlib inline

from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
import kerastuner as kt
print(tf.__version__)

##### Set random seeds to make your results reproducible

In [ ]:
# 매번 모델링을 할 때마다 동일한 결과를 얻으려면 아래 코드를 실행해야 함.

def reset_seeds(s1,s2,s3, reset_graph_with_backend=None):
    if reset_graph_with_backend is not None:
        K = reset_graph_with_backend
        K.clear_session()
        tf.compat.v1.reset_default_graph()
        print("KERAS AND TENSORFLOW GRAPHS RESET")  # optional

    np.random.seed(s1)
    random.seed(s2)
    tf.compat.v1.set_random_seed(s3)
    os.environ['CUDA_VISIBLE_DEVICES'] = ''  # for GPU
#    print("RANDOM SEEDS RESET")  # optional

### Step 1: Load and process the data

##### Read data

In [ ]:
# 학습데이터에 custid가 없으면 추가
IDtest = pd.DataFrame({'custid': pd.read_csv('data/머러컴피티션/X_test.csv', encoding = "cp949").custid.unique()})

In [ ]:
# 데이터불러오기

In [ ]:
df_train = pd.read_csv('snd_sum_shap_0615.csv')

In [ ]:
df_test = pd.read_csv('snd_sum_shap_te_0615.csv')

In [ ]:
y_train = pd.read_csv('data/머러컴피티션/y_train.csv').group

##### One-hot-encode Target variable 

In [ ]:
# 8개의 범주형 타겟 값을 one-hot-encoding을 통해 8개의 컬럼으로 만들어야 함.
y_train = keras.utils.to_categorical(y_train.astype('category').cat.codes)

In [ ]:
df_train

In [ ]:
df_trains = df_train.iloc[:,1:]
df_tests = df_test.iloc[:,1:]

In [ ]:
X_train = df_train
X_test = df_test

##### Split data into train & validation set 

In [ ]:
i = int(round(X_train.shape[0] * 0.8,0))
X_valid, y_valid = X_train[i:], y_train[i:]
X_train, y_train = X_train[:i], y_train[:i]

In [ ]:
y_train

##### Feature scaling

In [ ]:
X_train.shape, X_valid.shape, X_test.shape

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)

### Step 2: Define the hyper-model

In [ ]:
def model_fn(hp):
    inputs = keras.Input(shape=(X_train.shape[1],))
    x = inputs
    for i in range(hp.Int('num_layers', 2, 3)):
        x = keras.layers.Dense(hp.Int('unit_'+str(i), 16, 64, step=16), activation='relu')(x)
        x = keras.layers.Dropout(hp.Float('dropout_'+str(i), 0, 0.5, step=0.25, default=0.5))(x)
    outputs = keras.layers.Dense(8, activation='softmax')(x) # 예측값이 8종류이므로 8개 출력 뉴런 필요
    model = keras.Model(inputs, outputs)
    model.compile(loss='categorical_crossentropy', # Multiclass Classification에서 사용하는 loss function
                  optimizer=tf.keras.optimizers.Adam(hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])), 
                  metrics=[keras.metrics.CategoricalCrossentropy()]) # Multiclass Classification에서 사용하는 평가지표
    return model

### Step 3: Build multiple hyper-tuned models

In [ ]:
# 
tuner = kt.Hyperband(model_fn,
                     objective=kt.Objective('val_categorical_crossentropy', direction="min"), 
                     max_epochs=20,
                     hyperband_iterations=2,
                     overwrite=True,
                     directory='dnn_tuning')
tuner.search(X_train, y_train, validation_data=(X_valid, y_valid), 
             callbacks=[tf.keras.callbacks.EarlyStopping(patience=1)])
model = tuner.get_best_models(1)[0]  
tuner.results_summary(1)

In [ ]:
t = pd.Timestamp.now()
fname = f"dnn_submission_{t.month:02}{t.day:02}.1.492.csv"
pred = pd.DataFrame(model.predict(X_test))
pred.columns = ['F20','F30','F40','F50','M20','M30','M40','M50']

pd.concat([pd.Series(IDtest.custid, name="ID"), pred] ,axis=1).to_csv('data/머러컴피티션/진짜마지막/dnn결과/'+fname, index=False)
print(f"'{fname}' is ready to submit.")

In [ ]:
# 
tuner = kt.Hyperband(model_fn,
                     objective=kt.Objective('val_categorical_crossentropy', direction="min"), 
                     max_epochs=20,
                     hyperband_iterations=2,
                     overwrite=True,
                     directory='dnn_tuning')
tuner.search(X_train, y_train, validation_data=(X_valid, y_valid), 
             callbacks=[tf.keras.callbacks.EarlyStopping(patience=1)])
model = tuner.get_best_models(1)[0]  
tuner.results_summary(1)

In [ ]:
model = tuner.get_best_models(1)[0]  
tuner.results_summary(1)

In [ ]:
t = pd.Timestamp.now()
fname = f"dnn_submission_{t.month:02}{t.day:02}.1.481.csv"
pred = pd.DataFrame(model.predict(X_test))
pred.columns = ['F20','F30','F40','F50','M20','M30','M40','M50']

pd.concat([pd.Series(IDtest.custid, name="ID"), pred] ,axis=1).to_csv('data/머러컴피티션/submission/snd_sum_shap_0612.csv/'+fname, index=False)
print(f"'{fname}' is ready to submit.")

In [ ]:
# 
tuner = kt.Hyperband(model_fn,
                     objective=kt.Objective('val_categorical_crossentropy', direction="min"), 
                     max_epochs=20,
                     hyperband_iterations=2,
                     overwrite=True,
                     directory='dnn_tuning')
tuner.search(X_train, y_train, validation_data=(X_valid, y_valid), 
             callbacks=[tf.keras.callbacks.EarlyStopping(patience=1)])
model = tuner.get_best_models(1)[0]  
tuner.results_summary(1)

In [ ]:
t = pd.Timestamp.now()
fname = f"dnn_submission_{t.month:02}{t.day:02}.1.490.csv"
pred = pd.DataFrame(model.predict(X_test))
pred.columns = ['F20','F30','F40','F50','M20','M30','M40','M50']

pd.concat([pd.Series(IDtest.custid, name="ID"), pred] ,axis=1).to_csv('data/머러컴피티션/submission/snd_sum_shap_0612.csv/'+fname, index=False)
print(f"'{fname}' is ready to submit.")

In [ ]:
dnn487.to_csv('data/머러컴피티션/submission/snd_sum_shap_0612.csv/'+'dnn487', index=False)

In [ ]:
# second tuning 전 step=0.25,max_epochs=20, num 4 3 16 48 6
tuner = kt.Hyperband(model_fn,
                     objective=kt.Objective('val_categorical_crossentropy', direction="min"), 
                     max_epochs=20,
                     hyperband_iterations=2,
                     overwrite=True,
                     directory='dnn_tuning')
tuner.search(X_train, y_train, validation_data=(X_valid, y_valid), 
             callbacks=[tf.keras.callbacks.EarlyStopping(patience=1)])
model = tuner.get_best_models(1)[0]  
tuner.results_summary(1)

In [ ]:
t = pd.Timestamp.now()
fname = f"dnn_submission_{t.month:02}{t.day:02}.1.490.csv"
pred = pd.DataFrame(model.predict(X_test))
pred.columns = ['F20','F30','F40','F50','M20','M30','M40','M50']

In [ ]:

pd.concat([pd.Series(IDtest.custid, name="ID"), pred] ,axis=1).to_csv('data/머러컴피티션/submission/snd_sum_shap_0612.csv/'+fname, index=False)
print(f"'{fname}' is ready to submit.")

In [ ]:
# second tuning 전 step=0.25,max_epochs=20, num 3 80
tuner = kt.Hyperband(model_fn,
                     objective=kt.Objective('val_categorical_crossentropy', direction="min"), 
                     max_epochs=20,
                     hyperband_iterations=2,
                     overwrite=True,
                     directory='dnn_tuning')
tuner.search(X_train, y_train, validation_data=(X_valid, y_valid), 
             callbacks=[tf.keras.callbacks.EarlyStopping(patience=1)])
model = tuner.get_best_models(1)[0]  
tuner.results_summary(1)

In [ ]:
t = pd.Timestamp.now()
fname = f"dnn_submission_{t.month:02}{t.day:02}.1.489.csv"
pred = pd.DataFrame(model.predict(X_test))
pred.columns = ['F20','F30','F40','F50','M20','M30','M40','M50']
pd.concat([pd.Series(IDtest.custid, name="ID"), pred] ,axis=1).to_csv('data/머러컴피티션/submission/snd_sum_shap_0612.csv/'+fname, index=False)
print(f"'{fname}' is ready to submit.")

In [ ]:
# second tuning 전 step=0.25,max_epochs=20,
tuner = kt.Hyperband(model_fn,
                     objective=kt.Objective('val_categorical_crossentropy', direction="min"), 
                     max_epochs=20,
                     hyperband_iterations=2,
                     overwrite=True,
                     directory='dnn_tuning')
tuner.search(X_train, y_train, validation_data=(X_valid, y_valid), 
             callbacks=[tf.keras.callbacks.EarlyStopping(patience=1)])
model = tuner.get_best_models(1)[0]  
tuner.results_summary(1)

In [ ]:
# second tuning 전 step=0.2,max_epochs=20,
tuner = kt.Hyperband(model_fn,
                     objective=kt.Objective('val_categorical_crossentropy', direction="min"), 
                     max_epochs=20,
                     hyperband_iterations=2,
                     overwrite=True,
                     directory='dnn_tuning')
tuner.search(X_train, y_train, validation_data=(X_valid, y_valid), 
             callbacks=[tf.keras.callbacks.EarlyStopping(patience=1)])
model = tuner.get_best_models(1)[0]  
tuner.results_summary(1)

In [ ]:
# second tuning 전 step=0.25,max_epochs=20,
tuner = kt.Hyperband(model_fn,
                     objective=kt.Objective('val_categorical_crossentropy', direction="min"), 
                     max_epochs=20,
                     hyperband_iterations=2,
                     overwrite=True,
                     directory='dnn_tuning')
tuner.search(X_train, y_train, validation_data=(X_valid, y_valid), 
             callbacks=[tf.keras.callbacks.EarlyStopping(patience=1)])
model = tuner.get_best_models(1)[0]  
tuner.results_summary(1)

In [ ]:
# second tuning 후 step=0.25,max_epochs=20,
tuner = kt.Hyperband(model_fn,
                     objective=kt.Objective('val_categorical_crossentropy', direction="min"), 
                     max_epochs=20,
                     hyperband_iterations=2,
                     overwrite=True,
                     directory='dnn_tuning')
tuner.search(X_train, y_train, validation_data=(X_valid, y_valid), 
             callbacks=[tf.keras.callbacks.EarlyStopping(patience=1)])
model = tuner.get_best_models(1)[0]  
tuner.results_summary(1)

In [ ]:
# second tuning 후 256
tuner = kt.Hyperband(model_fn,
                     objective=kt.Objective('val_categorical_crossentropy', direction="min"), 
                     max_epochs=10,
                     hyperband_iterations=2,
                     overwrite=True,
                     directory='dnn_tuning')
tuner.search(X_train, y_train, validation_data=(X_valid, y_valid), 
             callbacks=[tf.keras.callbacks.EarlyStopping(patience=1)])
model = tuner.get_best_models(1)[0]  
tuner.results_summary(1)

In [ ]:
# 1st tuning
tuner = kt.Hyperband(model_fn,
                     objective=kt.Objective('val_categorical_crossentropy', direction="min"), 
                     max_epochs=20,
                     hyperband_iterations=2,
                     overwrite=True,
                     directory='dnn_tuning')
tuner.search(X_train, y_train, validation_data=(X_valid, y_valid), 
             callbacks=[tf.keras.callbacks.EarlyStopping(patience=1)])
model = tuner.get_best_models(1)[0]  
tuner.results_summary(1)

In [ ]:
X_train

In [ ]:
t = pd.Timestamp.now()
fname = f"dnn_submission_{t.month:02}{t.day:02}{t.hour:02}1.493.csv"
pred = pd.DataFrame(model.predict(X_test))
pred.columns = ['F20','F30','F40','F50','M20','M30','M40','M50']
pd.concat([pd.Series(IDtest.custid, name="ID"), pred] ,axis=1).to_csv('data/머러컴피티션/submission/snd_sum_shap_0612.csv/'+fname, index=False)
print(f"'{fname}' is ready to submit.")

In [ ]:
pd.read_csv('data/머러컴피티션/submission/dnn+lgbm/dnn_submission_06131301.csv')

In [ ]:
pd.read_csv('data/머러컴피티션/submission/dnn+lgbm/dnn_submission_0613_1.512.csv')

In [ ]:
pd.concat([pd.Series(IDtest.custid, name="ID"), pred] ,axis=1)

<font color="#CC3D3D"><p>
# End